In [46]:
import pickle
import os
import copy
import matplotlib.pyplot as plt
import numpy as np
import paper_style  # apply the style automatically
import gc
import psutil
import yaml
import sys
import torch
import io
from matplotlib.lines import Line2D
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns




# Input Config

In [47]:
df = pd.read_csv('/home/evrond/rl_for_curobo_analysis/projects_root/experiments/armsResults_date1409_rows960_rowTypeTrial.csv')
df

,sim_id,alg,task_type,task_level,task_seed,col_chance,task_val
0,2025-09-08_03:30:27_R_ur5e_N4_AO_Tbin_s5_l2,"O(400, 1)",bin,2,5,0.000,6.000000
1,2025-09-08_04:01:24_R_ur5e_N4_AO_Tbin_s3_l4,"O(400, 1)",bin,4,3,0.022,7.000000
2,2025-09-08_03:27:28_R_ur5e_N4_AO_Tbin_s4_l2,"O(400, 1)",bin,2,4,0.000,6.000000
3,2025-09-08_04:23:09_R_ur5e_N4_AO_Tbin_s4_l5,"O(400, 1)",bin,5,4,0.000,8.000000
4,2025-09-08_04:04:41_R_ur5e_N4_AO_Tbin_s4_l4,"O(400, 1)",bin,4,4,0.000,6.000000
...,...,...,...,...,...,...,...
955,2025-09-08_22:54:49_R_ur5e_N4_ACC_Tfollow_s5_l2,CC(500),follow,2,5,0.000,0.304553
956,2025-09-08_22:41:53_R_ur5e_N4_ACC_Tfollow_s2_l2,CC(500),follow,2,2,0.000,0.297082
957,2025-09-08_22:22:11_R_ur5e_N4_ACC_Tfollow_s3_l1,CC(500),follow,1,3,0.000,0.310030
958,2025-09-08_23:40:16_R_ur5e_N4_ACC_Tfollow_s3_l4,CC(500),follow,4,3,0.278,0.307996


In [129]:
def normalize(df, baseline_alg='O(400, 5)', normalization_type='diff_from_baseline',add_baseline_raw_vals=True):
    """
    normalization_type: "diff_from_baseline" or "ratio_from_max"
        diff_from_baseline: normalized_task_val = task_val - baseline_alg's task_val at the same (task_type, task_seed, task_level)
        ratio_from_max: normalized_task_val = task_val / max task_val over all algs at the same(task_type, task_seed, task_level)
    
    """
    
    df_work = df.copy()
    df_work['normalized_task_val'] = 0.0
    df_work['normalized_col_chance'] = 0.0
    if add_baseline_raw_vals:
        df_work['baseline_task_val'] = 0.0
        df_work['baseline_col_chance'] = 0.0
    
    # Get all unique combinations of task_type, task_seed, task_level
    unique_combinations = df_work[['task_type', 'task_seed', 'task_level']].drop_duplicates() # df with all unique combinations of task_type, task_seed, task_level, each appears only once
    
    for idx, row in unique_combinations.iterrows():
        # print(idx, row)
        task_type = row['task_type']
        task_seed = row['task_seed'] 
        task_level = row['task_level']
        
        # Find the baseline_alg rows with matching task_type, task_seed, task_level
        baseline_alg_mask = (
            (df_work['alg'] == baseline_alg) & 
            (df_work['task_type'] == task_type) & 
            (df_work['task_seed'] == task_seed) & 
            (df_work['task_level'] == task_level)
        )
        
        baseline_alg_rows = df_work[baseline_alg_mask]
        
        # Sanity check: verify there is exactly one baseline_alg row
        if len(baseline_alg_rows) == 0:
            raise ValueError(f"No {baseline_alg} row found for task_type={task_type}, "
                           f"task_seed={task_seed}, task_level={task_level}")
        elif len(baseline_alg_rows) > 1:
            raise ValueError(f"Multiple {baseline_alg} rows found for task_type={task_type}, "
                           f"task_seed={task_seed}, task_level={task_level}. Found {len(baseline_alg_rows)} rows.")
        
        # Get the baseline_alg task_val
        baseline_alg_task_val = baseline_alg_rows['task_val'].iloc[0]
        baseline_alg_col_chance = baseline_alg_rows['col_chance'].iloc[0]
        
            
        # Compute deviation for all rows with this combination
        combination_mask = (
            (df_work['task_type'] == task_type) & 
            (df_work['task_seed'] == task_seed) & 
            (df_work['task_level'] == task_level)
        )
        
        if normalization_type == 'diff_from_baseline':
            # normalize task_val - diff from baseline
            df_work.loc[combination_mask, 'normalized_task_val'] = (
                    df_work.loc[combination_mask, 'task_val'] - baseline_alg_task_val
                )
            

            # normalize col_chance - diff from baseline
            df_work.loc[combination_mask, 'normalized_col_chance'] = (
                    df_work.loc[combination_mask, 'col_chance'] - baseline_alg_col_chance
                )
            
        elif normalization_type == 'ratio_from_max':

            # normalize task_val - ratio from max
            max_task_val = df_work.loc[combination_mask, 'task_val'].max()
            if max_task_val == 0:
                normalized_ratio = 1 # avoid division by zero
            else:  
                normalized_ratio = df_work.loc[combination_mask, 'task_val'] / max_task_val  
            df_work.loc[combination_mask, 'normalized_task_val'] = normalized_ratio
            
            # normalize col_chance - ratio from max
            max_col_chance = df_work.loc[combination_mask, 'col_chance'].max()
            if max_col_chance == 0:
                normalized_col_chance = 1 # avoid division by zero
            else:  
                normalized_col_chance = df_work.loc[combination_mask, 'col_chance'] / max_col_chance
            df_work.loc[combination_mask, 'normalized_col_chance'] = normalized_col_chance
            
        if add_baseline_raw_vals:
            df_work.loc[combination_mask, 'baseline_task_val'] = baseline_alg_task_val
            df_work.loc[combination_mask, 'baseline_col_chance'] = baseline_alg_col_chance
                
    return df_work

In [130]:
df_normalized_diff_from_O400_5 = normalize(df)
df_normalized_ratio_from_max = normalize(df, normalization_type='ratio_from_max')




In [121]:
df_normalized_diff_from_O400_5

,sim_id,alg,task_type,task_level,task_seed,col_chance,task_val,normalized_task_val,normalized_col_chance,baseline_task_val,baseline_col_chance
0,2025-09-08_03:30:27_R_ur5e_N4_AO_Tbin_s5_l2,"O(400, 1)",bin,2,5,0.000,6.000000,1.000000,0.000,5.000000,0.000
1,2025-09-08_04:01:24_R_ur5e_N4_AO_Tbin_s3_l4,"O(400, 1)",bin,4,3,0.022,7.000000,0.000000,0.022,7.000000,0.000
2,2025-09-08_03:27:28_R_ur5e_N4_AO_Tbin_s4_l2,"O(400, 1)",bin,2,4,0.000,6.000000,-2.000000,0.000,8.000000,0.000
3,2025-09-08_04:23:09_R_ur5e_N4_AO_Tbin_s4_l5,"O(400, 1)",bin,5,4,0.000,8.000000,-1.000000,0.000,9.000000,0.000
4,2025-09-08_04:04:41_R_ur5e_N4_AO_Tbin_s4_l4,"O(400, 1)",bin,4,4,0.000,6.000000,-1.000000,0.000,7.000000,0.000
...,...,...,...,...,...,...,...,...,...,...,...
955,2025-09-08_22:54:49_R_ur5e_N4_ACC_Tfollow_s5_l2,CC(500),follow,2,5,0.000,0.304553,0.208860,0.000,0.095693,0.000
956,2025-09-08_22:41:53_R_ur5e_N4_ACC_Tfollow_s2_l2,CC(500),follow,2,2,0.000,0.297082,0.181914,0.000,0.115167,0.000
957,2025-09-08_22:22:11_R_ur5e_N4_ACC_Tfollow_s3_l1,CC(500),follow,1,3,0.000,0.310030,0.216714,0.000,0.093316,0.000
958,2025-09-08_23:40:16_R_ur5e_N4_ACC_Tfollow_s3_l4,CC(500),follow,4,3,0.278,0.307996,0.202744,0.278,0.105252,0.000


In [ ]:
# df_normalized_diff_from_O400_5_gb_task = df_normalized_diff_from_O400_5.groupby(['task_type'])
# df_normalized_diff_from_O400_5_gb_task_and_level = df_normalized_diff_from_O400_5.groupby(['task_type', 'task_level'])


In [131]:
def compute_algorithm_statistics(df, group_keys):
    """
    Compute mean and standard deviation statistics for each algorithm within groups.
    
    Groups the dataframe by the specified keys, then further groups by 'alg' within each group,
    and computes mean and std for normalized_task_val and normalized_col_chance.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing the data with columns: 'alg', 'normalized_task_val', 'normalized_col_chance'
        and the grouping keys
    group_keys : list
        List of column names to group by (e.g., ['task_type', 'task_level'])
        
    Returns:
    --------
    pandas.DataFrame
        Summary table with columns:
        - All group_keys columns
        - 'alg' column
        - 'normalized_task_val_mean', 'normalized_task_val_std'
        - 'normalized_col_chance_mean', 'normalized_col_chance_std'
        - 'count' (number of rows for each algorithm-group combination)
    """
    # Group by the specified keys and 'alg'
    grouped = df.groupby(group_keys + ['alg'])
    
    
    if not 'baseline_task_val' in df.columns and not 'baseline_col_chance' in df.columns:
        # Compute statistics for each group
        stats = grouped.agg({
            'normalized_task_val': ['mean', 'std'],
            'normalized_col_chance': ['mean', 'std'],
            'sim_id': 'count'  # Count number of rows
        }).round(3)
        
        # Flatten column names
        stats.columns = [
            'normalized_task_val_mean', 'normalized_task_val_std',
            'normalized_col_chance_mean', 'normalized_col_chance_std',
            'count'
        ]
    else:
        stats = grouped.agg({
            'normalized_task_val': ['mean', 'std'],
            'normalized_col_chance': ['mean', 'std'],
            'baseline_task_val': ['mean', 'std'],
            'baseline_col_chance': ['mean', 'std'],
            'sim_id': 'count'  # Count number of rows
        }).round(3)
        stats.columns = [
            'normalized_task_val_mean', 'normalized_task_val_std',
            'normalized_col_chance_mean', 'normalized_col_chance_std',
            'baseline_task_val_mean', 'baseline_task_val_std',
            'baseline_col_chance_mean','baseline_col_chance_std',
            'count'
        ]
    # Reset index to make group_keys and 'alg' regular columns
    stats = stats.reset_index()
    
    return stats

In [ ]:
# Group by task_type and task_level
result = compute_algorithm_statistics(df_normalized_diff_from_O400_5, ['task_type'])
# result.to_csv('df_normalized_diff_from_O400_5_grouped_by_task_baseline_raw_vals.csv', index=False)

In [127]:
result

,task_type,alg,normalized_task_val_mean,normalized_task_val_std,normalized_col_chance_mean,normalized_col_chance_std,baseline_task_val_mean,baseline_task_val_std,baseline_col_chance_mean,baseline_col_chance_std,count
0,bin,CC(1000),-6.033,3.146,0.010,0.075,6.500,2.596,0.004,0.011,30
1,bin,CC(500),-6.000,3.118,-0.004,0.011,6.500,2.596,0.004,0.011,30
2,bin,"O(400, 1)",-0.733,2.083,0.004,0.016,6.500,2.596,0.004,0.011,30
3,bin,"O(400, 5)",0.000,0.000,0.000,0.000,6.500,2.596,0.004,0.011,30
4,bin,"O-(400, 5)",-3.867,3.739,0.027,0.041,6.500,2.596,0.004,0.011,30
5,bin,"SC(1600, 5)",-2.167,3.119,0.080,0.080,6.500,2.596,0.004,0.011,30
6,bin,"SC(400, 5)",-4.733,2.638,0.045,0.055,6.500,2.596,0.004,0.011,30
7,bin,"SD(400, 5)",0.433,2.944,0.229,0.277,6.500,2.596,0.004,0.011,30
8,follow,CC(1000),0.201,0.025,0.119,0.150,0.104,0.020,0.006,0.020,30
9,follow,CC(500),0.197,0.026,0.120,0.150,0.104,0.020,0.006,0.020,30


In [ ]:
# Group by task_type and task_level
result = compute_algorithm_statistics(df_normalized_diff_from_O400_5, ['task_type', 'task_level'])
# result.to_csv('df_normalized_diff_from_O400_5_grouped_by_task_and_level_baseline_raw_vals.csv', index=False)

In [ ]:
result = compute_algorithm_statistics(df_normalized_ratio_from_max, ['task_type'])
# result.to_csv('df_normalized_ratio_from_max_grouped_by_task_baseline_raw_vals.csv', index=False)


In [ ]:
result = compute_algorithm_statistics(df_normalized_ratio_from_max, ['task_type','task_level'])
# result.to_csv('df_normalized_ratio_from_max_grouped_by_task_and_level_baseline_raw_vals.csv', index=False)

In [85]:
tmp = df_normalized_ratio_from_max.drop(columns=['sim_id'], inplace=False)

In [ ]:
def filter_task_level_seed(df, task_type, task_level, task_seed):
    return df[(df['task_type'] == task_type) & (df['task_level'] == task_level) & (df['task_seed'] == task_seed)]

bin10_dif = filter_task_level_seed(df_normalized_diff_from_O400_5, 'bin',1, 0)

In [ ]:
bin10_dif

,sim_id,alg,task_type,task_level,task_seed,col_chance,task_val,normalized_task_val,normalized_col_chance
14,2025-09-08_02:56:16_R_ur5e_N4_AO_Tbin_s0_l1,"O(400, 1)",bin,1,0,0.000,6.0,1.0,0.000
52,2025-09-06_21:35:18_R_ur5e_N4_AO_Tbin_s0_l1,"O(400, 5)",bin,1,0,0.000,5.0,0.0,0.000
64,2025-09-08_22:22:35_R_ur5e_N4_AO-_Tbin_s0_l1,"O-(400, 5)",bin,1,0,0.000,3.0,-2.0,0.000
96,2025-09-08_23:22:28_R_ur5e_N4_ASC_Tbin_s0_l1,"SC(1600, 5)",bin,1,0,0.048,1.0,-4.0,0.048
130,2025-09-08_22:48:40_R_ur5e_N4_ASC_Tbin_s0_l1,"SC(400, 5)",bin,1,0,0.000,2.0,-3.0,0.000
160,2025-09-07_01:29:01_R_ur5e_N4_ASD_Tbin_s0_l1,"SD(400, 5)",bin,1,0,0.000,3.0,-2.0,0.000
195,2025-09-09_08:22:34_R_ur5e_N4_ACC_Tbin_s0_l1,CC(1000),bin,1,0,0.000,2.0,-3.0,0.000
215,2025-09-09_00:13:00_R_ur5e_N4_ACC_Tbin_s0_l1,CC(500),bin,1,0,0.000,1.0,-4.0,0.000


In [ ]:
bin50_dif = filter_task_level_seed(df_normalized_diff_from_O400_5, 'bin',5, 0)
bin50_dif

,sim_id,alg,task_type,task_level,task_seed,col_chance,task_val,normalized_task_val,normalized_col_chance
26,2025-09-08_04:10:39_R_ur5e_N4_AO_Tbin_s0_l5,"O(400, 1)",bin,5,0,0.000,5.0,-2.0,-0.024
53,2025-09-06_22:41:03_R_ur5e_N4_AO_Tbin_s0_l5,"O(400, 5)",bin,5,0,0.024,7.0,0.0,0.000
63,2025-09-08_23:10:09_R_ur5e_N4_AO-_Tbin_s0_l5,"O-(400, 5)",bin,5,0,0.026,1.0,-6.0,0.002
112,2025-09-09_00:25:18_R_ur5e_N4_ASC_Tbin_s0_l5,"SC(1600, 5)",bin,5,0,0.080,4.0,-3.0,0.056
147,2025-09-08_23:45:12_R_ur5e_N4_ASC_Tbin_s0_l5,"SC(400, 5)",bin,5,0,0.076,2.0,-5.0,0.052
161,2025-09-07_02:13:41_R_ur5e_N4_ASD_Tbin_s0_l5,"SD(400, 5)",bin,5,0,0.202,11.0,4.0,0.178
183,2025-09-09_09:10:07_R_ur5e_N4_ACC_Tbin_s0_l5,CC(1000),bin,5,0,0.000,0.0,-7.0,-0.024
231,2025-09-09_01:13:10_R_ur5e_N4_ACC_Tbin_s0_l5,CC(500),bin,5,0,0.000,0.0,-7.0,-0.024


In [105]:
bin50_ratio = filter_task_level_seed(df_normalized_ratio_from_max, 'bin',5, 0)
bin50_ratio

,sim_id,alg,task_type,task_level,task_seed,col_chance,task_val,normalized_task_val,normalized_col_chance
26,2025-09-08_04:10:39_R_ur5e_N4_AO_Tbin_s0_l5,"O(400, 1)",bin,5,0,0.000,5.0,0.454545,0.000000
53,2025-09-06_22:41:03_R_ur5e_N4_AO_Tbin_s0_l5,"O(400, 5)",bin,5,0,0.024,7.0,0.636364,0.118812
63,2025-09-08_23:10:09_R_ur5e_N4_AO-_Tbin_s0_l5,"O-(400, 5)",bin,5,0,0.026,1.0,0.090909,0.128713
112,2025-09-09_00:25:18_R_ur5e_N4_ASC_Tbin_s0_l5,"SC(1600, 5)",bin,5,0,0.080,4.0,0.363636,0.396040
147,2025-09-08_23:45:12_R_ur5e_N4_ASC_Tbin_s0_l5,"SC(400, 5)",bin,5,0,0.076,2.0,0.181818,0.376238
161,2025-09-07_02:13:41_R_ur5e_N4_ASD_Tbin_s0_l5,"SD(400, 5)",bin,5,0,0.202,11.0,1.000000,1.000000
183,2025-09-09_09:10:07_R_ur5e_N4_ACC_Tbin_s0_l5,CC(1000),bin,5,0,0.000,0.0,0.000000,0.000000
231,2025-09-09_01:13:10_R_ur5e_N4_ACC_Tbin_s0_l5,CC(500),bin,5,0,0.000,0.0,0.000000,0.000000
